## Load Champion Model and Frozen Threshold

In [1]:
import pandas as pd
import joblib
import json

# Load dataset
df = pd.read_csv("data/creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

# Recreate the exact same split
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Load frozen model
lgb_model = joblib.load("models/lgbm_champion.pkl")

# Load frozen threshold
with open("models/lgbm_threshold.json", "r") as f:
    threshold_data = json.load(f)

lgb_best_threshold = threshold_data["threshold"]
MAX_FPR = threshold_data["max_fpr"]

print("Model and threshold loaded successfully.")
print("Frozen threshold:", lgb_best_threshold)

Model and threshold loaded successfully.
Frozen threshold: 0.014554874923771423


## Apply Frozen Decision Policy to Test Set

In [2]:
# Generate test probabilities
lgb_test_probs = lgb_model.predict_proba(X_test)[:, 1]

# Apply frozen threshold
test_preds = (lgb_test_probs >= lgb_best_threshold).astype(int)

from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()

print("TP:", tp)

print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

TP: 62
FP: 18
FN: 12
TN: 42630


## Utility Model (Financial Impact) and Approval Rate

In [3]:
# ---- Business Assumptions (edit if you want) ----
AVG_FRAUD_LOSS = 120      # £ per fraud case (expected loss)
FALSE_POSITIVE_COST = 5   # £ per false positive (ops + customer friction)

total = tp + fp + fn + tn

approval_rate = (tn + fn) / total  # approved = predicted legitimate
fraud_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

# Utility: prevent TP fraud loss, pay FP friction, lose FN fraud
utility = (tp * AVG_FRAUD_LOSS) - (fp * FALSE_POSITIVE_COST) - (fn * AVG_FRAUD_LOSS)

print("Approval Rate:", round(approval_rate * 100, 4), "%")
print("Fraud Recall:", round(fraud_recall * 100, 2), "%")
print("Estimated Utility (£):", round(utility, 2))

Approval Rate: 99.8127 %
Fraud Recall: 83.78 %
Estimated Utility (£): 5910


## Business Interpretation – Utility and Operational Impact

The champion LightGBM model was evaluated under a strict operational constraint of FPR ≤ 0.2%, reflecting a conservative fraud control policy aimed at minimizing customer friction.

### Operational Performance

- Approval Rate: 99.81%
- Fraud Recall: 83.78%
- False Positive Rate: 0.042%

This indicates that the system maintains near-total customer approval stability while successfully intercepting the majority of fraudulent transactions.

Only 18 legitimate transactions were incorrectly flagged out of more than 42,000 legitimate cases, demonstrating strong control of customer disruption.

### Financial Impact

Under the assumed cost structure:

- Average fraud loss per missed fraud: £120
- Operational/customer friction cost per false positive: £5

The model generates an estimated net utility of £5,910 on the test sample.

This confirms that:

- Fraud losses prevented materially exceed operational friction costs.
- The decision threshold is economically rational.
- The system aligns with risk-adjusted profitability objectives.

### Strategic Assessment

The model achieves a balanced fraud strategy:

- High fraud capture
- Minimal legitimate transaction disruption
- Positive net financial impact

This demonstrates that the system is not optimized purely for predictive accuracy, but for business value under operational constraints.

The next step is to validate whether this threshold is utility-optimal under the FPR policy boundary.

## Threshold Utility Optimization Under FPR Constraint

In [4]:
import numpy as np
from sklearn.metrics import roc_curve

thresholds_to_test = np.linspace(0, 1, 500)

results = []

for th in thresholds_to_test:
    preds = (lgb_test_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    if fpr <= MAX_FPR:
        utility = (tp * AVG_FRAUD_LOSS) - (fp * FALSE_POSITIVE_COST) - (fn * AVG_FRAUD_LOSS)
        results.append((th, utility, fpr))

# Find best threshold under constraint
best_th, best_utility, best_fpr = max(results, key=lambda x: x[1])

print("Best Threshold Under Constraint:", round(best_th, 6))
print("Best Utility (£):", round(best_utility, 2))
print("FPR at Best Threshold:", round(best_fpr, 6))

Best Threshold Under Constraint: 0.002004
Best Utility (£): 5965
FPR at Best Threshold: 0.00129


## Threshold Optimization Analysis

A threshold sweep was conducted under the operational constraint of FPR ≤ 0.2% to determine the maximum achievable economic utility on the test set.

### Findings

- Optimal threshold under constraint: 0.002004  
- Maximum achievable utility: £5,965  
- FPR at optimal threshold: 0.129%

The previously frozen validation-derived threshold generated £5,910 in utility, only £55 below the theoretical maximum.

### Interpretation

This confirms:

- The validation-based threshold selection process generalizes effectively.
- The decision policy is economically near-optimal.
- The system is stable and not overly sensitive to small threshold changes.

The marginal improvement from post-hoc optimization is minimal, indicating that the deployed threshold is operationally robust.